In [ ]:
import os
import sys

# Kaggle dataset paths (read-only)
REPO_ROOT = '/kaggle/input/datasets/maedaky/hbreader-code290626/HBreader_code290626'
DATASET_ROOT = '/kaggle/input/datasets/maedaky/akjv-verses'
WORKING_DIR = '/kaggle/working'

# Add src to Python path
sys.path.insert(0, REPO_ROOT)

# Data paths
bibleInVersePath = os.path.join(DATASET_ROOT, 'akjv_verses.jsonl')

# Output paths (use working directory for any writes)
faiss_path = os.path.join(WORKING_DIR, 'faiss_index')

# Create output directories if needed
os.makedirs(faiss_path, exist_ok=True)

# Model and search config
index_name = 'bible_faiss_index'
top_k = 5
# embedding_model = 'BAAI/bge-large-en-v1.5'
# llm_model = 'mistralai/Mistral-7B-Instruct-v0.1'
use_gpu_faiss = True

# Debug prints
print('REPO_ROOT:', REPO_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('WORKING_DIR:', WORKING_DIR)
print('jsonl_path:', bibleInVersePath)
print('faiss_path:', faiss_path)
# print('embedding_model:', embedding_model)
# print('llm_model:', llm_model)
print('use_gpu_faiss:', use_gpu_faiss)

In [ ]:
from src.modules.data_processing import (
    read_text_file,
    clean_bible_text_header,
    save_cleaned_text_to_file,
    parse_bible_verses,
    save_documents_to_jsonl,
    load_documents_from_jsonl,
)
from src.modules.vector_db import (
    initialize_embeddings,
    build_and_save_faiss_vector_db,
    load_faiss_vectorstore,
)
from src.modules.rag_chain import (
    initialize_llm,
    get_prompt_template,
    build_rag_chain,
)

print('Modules imported successfully.')

In [ ]:
documents = load_documents_from_jsonl(bibleInVersePath)

print(f'Total documents loaded or generated: {len(documents)}')

In [ ]:
embeddingModelPath = "/kaggle/input/models/andreasbis/baai-bge-m3/transformers/default/1"

embeddings = initialize_embeddings(model_name=embeddingModelPath)
build_and_save_faiss_vector_db(documents, embeddings, faiss_path, index_name=index_name, use_gpu=use_gpu_faiss)

In [ ]:
retriever = load_faiss_vectorstore(faiss_path, embeddings, top_k=top_k, index_name=index_name, use_gpu=use_gpu_faiss)
test_query = 'How many days did God take to create the world?'
documents = retriever.get_relevant_documents(test_query)
print(f'Query: {test_query}')
for idx, doc in enumerate(documents, start=1):
    print(f'--- Result {idx} ---')
    print('Location:', doc.metadata.get('location'))
    print('Text:', doc.page_content)
    print()

In [ ]:
llmModelPath = "/kaggle/input/models/mistral-ai/mistral/pytorch/7b-instruct-v0.1-hf/1"

llm = initialize_llm(llmModelPath)
if llm is not None:
    prompt_template = get_prompt_template()
    rag_chain = build_rag_chain(retriever, llm, prompt_template)
    response = rag_chain(test_query)
    print('--- Generated Answer ---')
    print(response['answer'])
    print()
    print('--- Sources ---')
    for src in response['source_documents']:
        print(src.metadata.get('location'), ':', src.page_content)
else:
    print('LLM failed to initialize.')